# Romansh retrieval playground
Run the cells in order in JupyterLab from a clone of this repository. Install dependencies in the active environment with `python -m pip install -r requirements.txt jupyterlab ipywidgets` from the repository root.

Start by exploring the **La Quotidiana** cleaning audit, then try retrieval on aligned **Mediomatix** schoolbook segments, and finally compare the team's recorded held-out results. The newspaper corpus is a separate data-quality study. Its articles were not used for retrieval training or evaluation.

The audit and results use committed files. Live retrieval needs the Mediomatix validation split. TF-IDF runs on CPU; dense models are optional.

In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src" / "romansh_retrieval").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from a clone of the project repository.")
sys.path.insert(0, str(ROOT / "src"))

from functools import lru_cache
import ipywidgets as widgets
import numpy as np
import pandas as pd
from IPython.display import display
from romansh_retrieval.config import IDIOMS, DEFAULT_MODEL, XLM_R_MODEL
from romansh_retrieval.data import load_split, make_pairs
from romansh_retrieval.methods import tfidf_vectors, load_embedding_model, embedding_vectors

## 1. Explore the newspaper audit
Filter the manually reviewed sample by cleaning flag or variety. The counts describe the audit sample, which was stratified by flag.

In [ ]:
audit = pd.read_csv(ROOT / "data_quality" / "audit_sample_labeled.csv")

def audit_view(flag, variety):
    selected = audit
    if flag != "All":
        selected = selected[selected["flag"].eq(flag)]
    if variety != "All":
        selected = selected[selected["variety"].eq(variety)]
    print(f"{len(selected)} audited articles match")
    display(selected["verdict"].value_counts().rename("articles").to_frame())
    with pd.option_context("display.max_colwidth", 240):
        display(selected[["post_id", "variety", "flag", "verdict", "text"]].head(12))

display(widgets.interactive(
    audit_view,
    flag=widgets.Dropdown(options=["All", *sorted(audit["flag"].dropna().unique())]),
    variety=widgets.Dropdown(options=["All", *sorted(audit["variety"].dropna().unique())]),
))

## 2. Play with a live retrieval query
From the repository root, run `python scripts/fetch_mediomatix.py --splits validation` once. Choose a direction, a seeded candidate pool, and a query index within that pool. The table marks the aligned target so you can see when a plausible-looking alternative ranks ahead of it.

For MiniLM or pretrained XLM-R, install `requirements-embeddings.txt`. The first run downloads weights. To try the fine-tuned model live, train and save a checkpoint with [the fine-tuning notebook](finetuning_mediomatix.ipynb), then enter the path to its Hugging Face model directory. The playground applies the same masked mean pooling and 128-token limit as the fine-tuning notebook. Playground ranks use validation candidates. Section 3 shows the separate held-out benchmark.

In [ ]:
@lru_cache(maxsize=1)
def validation_data():
    return load_split("validation", ROOT / "data" / "raw")

@lru_cache(maxsize=2)
def pretrained_model(model_name):
    return load_embedding_model(model_name)

@lru_cache(maxsize=1)
def fine_tuned_model(model_path):
    from sentence_transformers import SentenceTransformer, models
    encoder = models.Transformer(model_path, max_seq_length=128)
    pooling = models.Pooling(encoder.get_word_embedding_dimension(), pooling_mode="mean")
    model = SentenceTransformer(modules=[encoder, pooling])
    model.eval()
    return model

@lru_cache(maxsize=4)
def retrieval_pool(source, target, pool_size, method, checkpoint):
    pairs = make_pairs(validation_data(), source, target, max_rows=pool_size, seed=13)
    if pairs.empty:
        raise ValueError("No aligned rows are available for this direction.")
    queries, candidates = pairs["src_text"].tolist(), pairs["tgt_text"].tolist()
    if method == "TF-IDF":
        q, c = tfidf_vectors(queries, candidates)
    elif method == "Fine-tuned XLM-R":
        q, c = embedding_vectors(queries, candidates, checkpoint,
                                 model=fine_tuned_model(checkpoint))
    else:
        name = DEFAULT_MODEL if method == "MiniLM" else XLM_R_MODEL
        q, c = embedding_vectors(queries, candidates, name, model=pretrained_model(name))
    return pairs, q, c

def retrieval_view(direction, pool_size, query, method, checkpoint, top_k):
    source, target = direction
    model_path = ""
    if method == "Fine-tuned XLM-R":
        if not checkpoint.strip():
            print("Enter the directory of a saved fine-tuned checkpoint.")
            return
        model_dir = Path(checkpoint.strip()).expanduser()
        if not model_dir.is_absolute():
            model_dir = ROOT / model_dir
        if not (model_dir / "config.json").is_file():
            print("Choose a local checkpoint directory containing config.json.")
            return
        model_path = str(model_dir.resolve())
    try:
        pairs, q, c = retrieval_pool(source, target, pool_size, method, model_path)
    except (FileNotFoundError, RuntimeError, ValueError, ImportError, OSError) as exc:
        print(exc)
        return
    if query >= len(pairs):
        print(f"Choose a query index from 0 to {len(pairs) - 1}.")
        return
    scores = q[query:query + 1] @ c.T
    scores = np.asarray(scores.toarray() if hasattr(scores, "toarray") else scores).ravel()
    gold = int(pairs.iloc[query]["gold"])
    gold_rank = 1 + np.count_nonzero(scores > scores[gold])
    order = np.argsort(-scores, kind="stable")[:top_k]
    print("Query:", pairs.iloc[query]["src_text"])
    print(f"Aligned target (rank {gold_rank}, ties share the best rank):",
          pairs.iloc[gold]["tgt_text"])
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame({"cosine": scores[order], "aligned_match": order == gold,
                              "candidate": pairs.iloc[order]["tgt_text"].to_numpy()}))

directions = [(f"{a} → {b}", (a, b)) for a in IDIOMS for b in IDIOMS if a != b]
display(widgets.interactive(
    retrieval_view, {"manual": True, "manual_name": "Rank candidates"},
    direction=widgets.Dropdown(options=directions),
    pool_size=widgets.Dropdown(options=[20, 100, 500], value=100),
    query=widgets.BoundedIntText(value=0, min=0, max=499),
    method=widgets.Dropdown(options=["TF-IDF", "MiniLM", "Pretrained XLM-R", "Fine-tuned XLM-R"]),
    checkpoint=widgets.Text(placeholder="Fine-tuned checkpoint directory"),
    top_k=widgets.IntSlider(min=1, max=10, value=5),
))

## 3. Compare the recorded held-out benchmark
Change the lexical overlap group and metric to compare TF-IDF, pretrained XLM-R, and fine-tuned XLM-R on the same held-out test protocol. These are recorded results from `results/test_summary.csv`, so this section runs without downloading models or retraining.

In [ ]:
test_summary = pd.read_csv(ROOT / "results" / "test_summary.csv")

def benchmark_view(bucket, metric):
    rows = test_summary[test_summary["bucket"].eq(bucket)].sort_values(metric, ascending=False)
    values = rows.set_index("model")[[metric, "directions", "queries"]]
    display(values.style.format({metric: "{:.2%}"}).bar(subset=[metric], vmin=0, vmax=1,
                                                        color="#90c4ed"))

display(widgets.interactive(
    benchmark_view,
    bucket=widgets.Dropdown(options=[("All", "all"), ("Low", "low"),
                                     ("Medium", "mid"), ("High", "high")]),
    metric=widgets.Dropdown(options=[("Recall@1", "recall_at_1"), ("MRR", "mrr")]),
))